# Polars CSE findings with Bartons indicators

These examples inspect Polars' optimized logical plan to see when common subexpression elimination (CSE) is applied. A generated column named `__POLARS_CSER_...` is evidence that expression CSE rewrote the projection. The exact generated name is an implementation detail.

In [ ]:
import polars as pl

from bartons.indicators import DMI, SMA, TYPPRICE

print(f"Polars {pl.__version__}")

prices = pl.DataFrame(
    {
        "symbol": ["A", "A", "A", "B", "B", "B"],
        "high": [11.0, 12.0, 13.0, 21.0, 22.0, 23.0],
        "low": [9.0, 10.0, 11.0, 19.0, 20.0, 21.0],
        "close": [10.0, 11.0, 12.0, 20.0, 21.0, 22.0],
    }
).lazy()

In [ ]:
def show_cse(query: pl.LazyFrame) -> str:
    plan = query.explain(optimized=True, format="tree")
    print(plan)
    print("Expression CSE temporary found:", "__POLARS_CSER_" in plan)
    return plan

## CSE applies: repeated native Bartons expression

`TYPPRICE` is implemented entirely from native Polars arithmetic. Two separately constructed but structurally equal expressions are deduplicated, demonstrating that Python object identity is not required.

In [ ]:
native_cse = prices.select(
    TYPPRICE().alias("typical_a"),
    TYPPRICE().alias("typical_b"),
)
native_cse_plan = show_cse(native_cse)

## CSE does not apply: repeated kernel-backed SMA

`SMA` is a stateful Bartons plugin kernel registered as non-elementwise. Reusing the same Python `Expr` does not create a CSE temporary in the current optimized plan.

In [ ]:
sma = SMA(2)
plugin_no_cse = prices.select(
    sma.alias("sma_a"),
    sma.alias("sma_b"),
)
plugin_no_cse_plan = show_cse(plugin_no_cse)

## CSE does not apply: fields from one reused DMI expression

The two projections contain the same underlying `dmi_expr`, but direct field extraction does not cause Polars to materialize that common plugin subtree. The optimized plan contains two textual `dmi_expr` occurrences and no CSE temporary.

In [ ]:
dmi = DMI(2)
fields_no_cse = prices.select(
    dmi.struct.field("adx"),
    dmi.struct.field("pdi"),
)
fields_no_cse_plan = show_cse(fields_no_cse)
print("dmi_expr occurrences:", fields_no_cse_plan.count("dmi_expr"))

## CSE does not cross `.over(...)`

Window expressions are explicit CSE traversal boundaries in Polars. Reusing the windowed struct expression therefore does not deduplicate its field projections.

In [ ]:
grouped_dmi = DMI(2).over("symbol")
over_no_cse = prices.select(
    grouped_dmi.struct.field("adx"),
    grouped_dmi.struct.field("pdi"),
)
over_no_cse_plan = show_cse(over_no_cse)

## Guaranteed sharing: materialize the struct once

An intermediate column creates an explicit dependency rather than relying on CSE. The optimized plan has one `dmi_expr` call below the field projection.

In [ ]:
materialized = (
    prices.with_columns(DMI(2).alias("_dmi"))
    .select(
        pl.col("_dmi").struct.field("adx"),
        pl.col("_dmi").struct.field("pdi"),
    )
)
materialized_plan = show_cse(materialized)
print("dmi_expr occurrences:", materialized_plan.count("dmi_expr"))
materialized.collect()

## Conclusions

- `__POLARS_CSER_...` in the optimized plan is the clearest quick evidence of expression CSE.
- Structurally equal native expressions can be deduplicated even when created as separate Python objects.
- Reusing one Python `Expr` does not guarantee shared evaluation for kernel-backed Bartons indicators.
- `.over(...)` prevents CSE traversal.
- Materialize a fused struct as an intermediate column when single execution is required.